# Exp01 RGB Baseline

GPU 런타임에서 저장소를 복제하고 `/content/UC-FoodWaste`에 둡니다. 마운트 후 아래 Drive 경로를 조정합니다. Smoke 결과를 검토하고 사용자 승인 후 정식 학습 셀을 실행합니다.

In [ ]:
%cd /content/UC-FoodWaste
%pip install -r requirements-yolo.txt

In [ ]:
from google.colab import drive
import subprocess
import sys

drive.mount("/content/drive")
opts = [
    "--data-root", "/content/drive/MyDrive/uc-foodwaste/data",
    "--prepared", "/content/ucfw-yolo",
    "--run-root", "/content/drive/MyDrive/uc-foodwaste-runs",
]

def exp01(stage, *extra):
    subprocess.run([sys.executable, "scripts/yolo/exp01.py", stage, *opts, *extra], check=True)

exp01("environment")

In [ ]:
from pathlib import Path

exp01("audit")
if not Path("/content/ucfw-yolo/dataset_summary.json").is_file():
    exp01("prepare")
exp01("smoke", "--batch", "2")

## 정식 학습

Smoke 및 평가 규칙 검토 후 실행합니다.

In [ ]:
exp01("train", "--batch", "16")
exp01("predict", "--split", "val")
exp01("threshold")
exp01("evaluate", "--split", "val")

## 최종 Test

평가 규칙과 Validation threshold가 확정된 뒤 이 셀을 실행합니다.

In [ ]:
exp01("predict", "--split", "test")
exp01("evaluate", "--split", "test")